# Setup

In [24]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Inizializza il client di Groq (richiede GROQ_API_KEY nelle variabili d'ambiente)
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# 2. Rileva automaticamente un modello valido presente sul tuo account
modelli_disponibili = [m.id for m in client.models.list().data]
print("Modelli disponibili sul tuo account Groq:", modelli_disponibili)

# Seleziona il primo modello Llama disponibile, oppure il primo in assoluto della lista
modello_scelto = next(
    (m for m in modelli_disponibili if "llama" in m.lower()),
    modelli_disponibili[0],
)
print(f"Modello selezionato automaticamente: '{modello_scelto}'\n")

def llm(prompt, stop=["\n"]):
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",  # Modello gratuito e velocissimo
        messages=[{"role": "user", "content": prompt}],
        stop=stop,
        temperature=0,
    )
    return response.choices[0].message.content

Modelli disponibili sul tuo account Groq: ['openai/gpt-oss-120b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-20b', 'whisper-large-v3', 'whisper-large-v3-turbo', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'allam-2-7b', 'canopylabs/orpheus-arabic-saudi']
Modello selezionato automaticamente: 'meta-llama/llama-prompt-guard-2-22m'



In [25]:
import wikienv, wrappers
env = wikienv.WikiEnv()
env = wrappers.HotPotQAWrapper(env, split="dev")
env = wrappers.LoggingWrapper(env)

def step(env, action):
    attempts = 0
    while attempts < 10:
        try:
            return env.step(action)
        except wikienv.requests.exceptions.Timeout:
            attempts += 1

# ReAct

In [ ]:
import json
import sys

folder = './prompts/'
prompt_file = 'prompts_naive.json'
with open(folder + prompt_file, 'r') as f:
    prompt_dict = json.load(f)

webthink_examples = prompt_dict['webthink_simple6']
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types: 
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.
"""
webthink_prompt = instruction + webthink_examples

def webthink(idx=None, prompt=webthink_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0

    for i in range(1, 8):
        n_calls += 1
        # Genera il pensiero e l'azione dall'LLM
        thought_action = llm(
            prompt + f"Thought {i}:", stop=[f"\nObservation {i}:"]
        )

        # 1. Parsing di Thought ed Action
        if f"\nAction {i}: " in thought_action:
            thought, action = thought_action.strip().split(f"\nAction {i}: ")
        elif "Action" in thought_action:
            # Gestisce variazioni come "Action 1:" senza spazio finale
            parts = thought_action.strip().split("Action")
            thought = parts[0].replace(f"Thought {i}:", "").strip()
            action = parts[1].split(":", 1)[-1].strip()
        else:
            # Se l'LLM ha dimenticato completamente l'etichetta 'Action:'
            n_badcalls += 1
            lines = thought_action.strip().split("\n")
            thought = lines[0].replace(f"Thought {i}:", "").strip()

            # Estraiamo l'ultima riga pulita da usare come risposta diretta
            risposta_estratta = (
                lines[-1]
                .replace("ohh...", "")
                .replace("**Answer:**", "")
                .replace("Answer:", "")
                .strip()
            )
            action = f"finish[{risposta_estratta}]"

        # 2. Normalizzazione del formato dell'Azione per l'ambiente ReAct
        action_formatted = action.strip()

        # Controlliamo se l'azione corrisponde a uno dei comandi standard (search, lookup, finish)
        if not (
            action_formatted.startswith("search[")
            or action_formatted.startswith("lookup[")
            or action_formatted.startswith("finish[")
        ):
            # Rimuove eventuali punti fermi finali o virgolette per non rovinare l'Exact Match (EM)
            pulita = action_formatted.strip(". \"'")
            action_formatted = f"finish[{pulita}]"

        # 3. Esecuzione dello step nell'ambiente
        obs, r, done, info = step(env, action_formatted)
        obs = str(obs).replace("\\n", "")

        step_str = f"Thought {i}: {thought}\nAction {i}: {action_formatted}\nObservation {i}: {obs}\n"
        prompt += step_str

        if to_print:
            print(step_str)
        if done:
            break

    # Se finiscono i 7 tentativi senza chiusura, forza un finish vuoto
    if not done:
        obs, r, done, info = step(env, "finish[]")

    if to_print:
        print(info, "\n")

    info.update(
        {"n_calls": n_calls, "n_badcalls": n_badcalls, "traj": prompt}
    )
    return r, info

In [32]:
import random
import time
idxs = list(range(7405))
random.Random(233).shuffle(idxs)

rs = []
infos = []
old_time = time.time()
for i in idxs[:500]:
    r, info = webthink(i, to_print=True)
    rs.append(info['em'])
    infos.append(info)
    print(sum(rs), len(rs), sum(rs) / len(rs), (time.time() - old_time) / len(rs))
    print('-----------')
    print()

3687 Question: What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?
ohh... Three Colors: Red (1993)
Thought 1: Three Colors: Red (1993)
Action 1: 
Observation 1: Episode finished, reward = 0


{'steps': 1, 'answer': '', 'gt_answer': 'Beyond the Clouds', 'question_idx': 3687, 'reward': False, 'em': False, 'f1': 0} 

0 1 0.0 11.15759539604187
-----------

6238 Question: Who created the show with Wendy Schaal doing the voice of Francine?
ohh... **Answer:** The show was created by **Seth MacFarlane** (co‑created with Mike Barker and Matt Weitzman).
Thought 1: **Answer:** The show was created by **Seth MacFarlane** (co‑created with Mike Barker and Matt Weitzman).
Action 1: 
Observation 1: Episode finished, reward = 0


{'steps': 1, 'answer': '', 'gt_answer': 'Seth MacFarlane', 'question_idx': 6238, 'reward': False, 'em': False, 'f1': 0} 

0 2 0.0 19.687056064605713
-----------

5388 Question: What objects were carried into battl

KeyboardInterrupt: 